# OpenRouter provider

Goal: probe `finstack_ai.providers.openrouter.is_available()`, construct `Agent.openrouter` for OpenRouter's stateless Responses endpoint, show `:nitro`/`:floor` model-routing suffixes, compose an explicit `OpenRouterMediaToolset`, then optionally run a live model and tool call with an explicit `api_key`.

Trust: T1 native provider plus T2 Python ports. Network: skipped unless the next cell or `OPENROUTER_API_KEY` has a key.

`api_key` is required and keyword-only. The binding does not read environment variables. Paste a key below, or keep using the environment.

Set `OPENROUTER_MODEL` to an OpenRouter model id. `reasoning_effort` is optional (`none`, `minimal`, `low`, `medium`, `high`, `xhigh`, `max`), and `reasoning_summary` is optional (`auto`, `concise`, `detailed`). `referer` and `title` set non-secret `HTTP-Referer`/`X-Title` attribution headers. Requests target `https://openrouter.ai/api/v1/responses` and omit `store` entirely (the request is stateless).

In [ ]:
OPENROUTER_MEDIA_API_KEY = ""  # separate media credential; construction only
OPENROUTER_API_KEY = ""  # paste a valid key to run the live cells; never commit it
OPENROUTER_MODEL = "openai/gpt-5"
OPENROUTER_REFERER = ""  # optional non-secret HTTP-Referer
OPENROUTER_TITLE = ""  # optional non-secret X-Title
OPENROUTER_REASONING_EFFORT = "low"  # or none / minimal / medium / high / xhigh / max
OPENROUTER_REASONING_SUMMARY = "auto"  # or concise / detailed

## Availability probe

`finstack_ai.providers` is a lazy subpackage; each linked provider exposes an `is_available()` probe backed by `linked_providers()`. This wheel links `openai`, `anthropic`, `gemini`, `ollama`, and `openrouter`, so both checks below should agree.

In [ ]:
import finstack_ai
import finstack_ai.providers as providers

print("openrouter is_available:", providers.openrouter.is_available())
print("linked_providers:", finstack_ai.linked_providers())

## Construction

The live cell uses `OPENROUTER_MODEL`, `OPENROUTER_REFERER`, `OPENROUTER_TITLE`, `OPENROUTER_REASONING_EFFORT`, and `OPENROUTER_REASONING_SUMMARY` from the first code cell. `compact_capability_catalog()` lists model-activated capabilities only, so it stays empty unless those are registered.

In [ ]:
from _support import live_value

api_key = live_value(OPENROUTER_API_KEY, "OPENROUTER_API_KEY")
if api_key:
    agent = await finstack_ai.Agent.openrouter(
        OPENROUTER_MODEL,
        "Answer concisely.",
        api_key=api_key,
        referer=OPENROUTER_REFERER or None,
        title=OPENROUTER_TITLE or None,
        reasoning_effort=OPENROUTER_REASONING_EFFORT or None,
        reasoning_summary=OPENROUTER_REASONING_SUMMARY or None,
    )
    print(agent.compact_capability_catalog() or "(no model-activated capabilities)")
else:
    print("skipped: OPENROUTER_API_KEY unset")

## Provider routing

OpenRouter's routing controls (`provider`, e.g. `{"order": ["openai"], "sort": "throughput"}`, or a `models` fallback array) pass through Rust `ModelSettings` untouched, but the Python binding does not expose a per-run settings argument — `Agent.run` takes `input`, `timeout_seconds`, `max_cycles`, `max_output_retries`, `capability`, and `attachments` only, with no `settings`/`model_settings` parameter. From Python, request routing through the `:nitro` (highest throughput) or `:floor` (lowest price) model-name suffixes instead, appended directly to the model id passed to `Agent.openrouter`.

In [ ]:
if api_key:
    nitro_agent = await finstack_ai.Agent.openrouter(
        f"{OPENROUTER_MODEL}:nitro",
        "Answer concisely.",
        api_key=api_key,
        reasoning_effort=OPENROUTER_REASONING_EFFORT or None,
        reasoning_summary=OPENROUTER_REASONING_SUMMARY or None,
    )
    print(
        nitro_agent.compact_capability_catalog() or "(no model-activated capabilities)"
    )
else:
    print("skipped: OPENROUTER_API_KEY unset")

## Media input

Media *input* (attaching images, PDFs, or audio to a user message so the
model can read them) is a host-side Rust concern, not a Python
constructor keyword. A host attaches a `MediaResolver` implementation
(ADR-049) to the provider config via `with_media_resolver`
(`OpenRouterConfig`, `OpenAIConfig`, `AnthropicConfig`,
`OllamaConfig`) to turn a kernel `BlobRef` into bytes or a URL the
provider puts on the wire; the Python binding does not expose this
hook, so there is no `Agent.openrouter(...)` keyword for it. Without a
configured resolver, a media-bearing user message fails closed
(`openrouter_request_invalid`) instead of being silently dropped. Each
provider also advertises per-model `InputCapabilities` toggles
(`with_input_images`, `with_input_audio`, `with_input_files`). For
OpenRouter the catalog fetch can set image and file flags from
`architecture.input_modalities`. Catalog audio stays off; hosts opt
in with `with_input_audio`. Modality support differs by provider:
OpenRouter and OpenAI accept images, files, and audio; Anthropic
accepts images and documents but no audio; Ollama accepts base64
images only.

**Audio caveat**: OpenRouter documents audio input only for
`/api/v1/chat/completions` (base64 `input_audio`, not URLs). The
OpenRouter provider crate maps `ContentBlock::Audio` onto the
Responses endpoint's `input_audio` item type by analogy, but whether
`/api/v1/responses` actually accepts `input_audio` is **unverified** —
it has not been confirmed against a live OpenRouter response. Hosts
should enable `with_input_audio` only after confirming the target
model accepts audio input on the Responses endpoint; audio resolved to
a URL (rather than inline bytes) is always rejected, since
OpenRouter's documented audio input is base64-only.

`OpenRouterMediaToolset(api_key=...)` configures media tools independently of the chat provider. Pass it through `toolsets=[media]` to share the agent artifact store. The following cells only construct agents and display the stable toolset identity; they do not execute tools or spend media credits. Media credentials are supplied separately.

In [ ]:
media = finstack_ai.OpenRouterMediaToolset(
    api_key=OPENROUTER_MEDIA_API_KEY or "offline-media-fixture",
    referer=OPENROUTER_REFERER or None,
    title=OPENROUTER_TITLE or None,
)
media_agent = await finstack_ai.Agent.openrouter(
    OPENROUTER_MODEL,
    "Answer concisely.",
    api_key=api_key or "offline-model-fixture",
    toolsets=[media],
    reasoning_effort=OPENROUTER_REASONING_EFFORT or None,
    reasoning_summary=OPENROUTER_REASONING_SUMMARY or None,
)
print("configured toolset:", media.component)

The same `media` object works with any model provider through `toolsets=[media]`. Its explicit media credential determines billing. This cell constructs a keyless Anthropic loopback agent without making a request.

In [ ]:
cross_provider_agent = await finstack_ai.Agent.anthropic(
    "http://127.0.0.1:9",
    "fixture-model",
    instruction="Answer concisely.",
    toolsets=[media],
)
print("same toolset, different model provider:", media.component)

## Live walkthrough

The rest of the notebook actually calls OpenRouter once per modality:
text, text plus tools, an image, a video, speech to text, and text to
speech. Every cell is gated on `api_key`, and each one spends real
credits on the key from the first code cell.

**Balance**: OpenRouter authorizes each request against the most it
could cost rather than what it ends up costing, so a nearly-spent key
fails before the model runs. `openai/gpt-5` advertises a 128k output
ceiling, and a balance that cannot cover 128k output tokens is refused
with `openrouter_http_error: ... HTTP 402: This request requires more
credits, or fewer max_tokens. You requested up to 128000 tokens, but can
only afford ...`. The remedy is to top the key up or to set
`OPENROUTER_MODEL` to a model whose ceiling the balance covers —
`openai/gpt-5-mini` is roughly a fifth the output price. Media models
are billed per image, clip, or job and are unaffected by this ceiling.

Two mechanics apply to every paid media call.

**Approval gate**: paid media generation (`openrouter_generate_image`,
`openrouter_generate_speech`, `openrouter_generate_video`,
`openrouter_transcribe_audio`) carries `ApprovalRequirement::Policy`,
and this SDK build maps that requirement to
`ToolPolicyDecision::RequireApproval`
(`crates/finstack-ai/src/agent/handle.rs`). `openrouter_get_video` does
not require approval. Before a paid tool executes, the run parks on a
durable `approval` interaction instead of returning a result — this is
by design: a host, not the model, decides whether a paid call proceeds.
A one-shot `await media_agent.run(...)` never resolves that interaction,
so it raises `RuntimeError: agent_run_runtime_failure: run did not
complete successfully`.

The fix is the same interaction flow notebook 10 (`10_elicitation.ipynb`)
uses for `ask_user`, just resolving an `approval` kind instead of
`free_text`/`form`: call `Agent.start` (not `Agent.run`) to get a `Run`
handle immediately, poll `run.list_interactions()` until the pending
approval appears, resolve it with
`run.resolve_interaction({..., "response": {"approved": True}})`, and
only then await `run.result()`.

**Generated bytes are artifacts, not model output**: an image or an
audio clip is hundreds of kilobytes of binary that a model cannot read.
Rather than inlining base64 into the conversation, the media toolset
stages the bytes in the agent's artifact store and puts only a
reference in the tool result. So the bytes never reach `result.text` —
the model only ever sees that a file exists. `Agent.read_artifact`
resolves one of those references back into `bytes`, which is how the
cells below write real files to disk and print URLs for them. Only
video has a remote URL, because OpenRouter hosts the rendered clip
itself; images and speech are local files.

In [ ]:
import asyncio
import json
import pathlib
import time

MEDIA_DIR = pathlib.Path("media-output")


def approval_resolution(pending: dict, approved: bool) -> dict:
    """Build one resolution for a parked paid-tool approval."""
    return {
        "interaction_id": pending["interaction_id"],
        "resolution_id": f"notebook-openrouter-{pending['interaction_id']}",
        "principal": {
            "issuer": "finstack-ai-python",
            "subject": "local-user",
            "tenant_scope": "python-local",
        },
        "authorization": {
            "policy_version": "python-policy-v1",
            "decision_id": "python-decision-v1",
        },
        "response": {"approved": approved},
    }


async def run_with_approvals(
    handle: "finstack_ai.Agent",
    prompt: str,
    *,
    budget: int = 1,
    timeout_seconds: float = 300.0,
    heartbeat: str = "",
) -> tuple["finstack_ai.RunResult", list[dict]]:
    """Run one prompt, approving at most `budget` paid tool calls.

    Returns the run result together with every JSON tool result the run
    produced, so a caller can take artifact references and URLs from the
    tool output rather than parsing them back out of model prose.
    """

    run = handle.start(prompt, timeout_seconds=timeout_seconds)
    tool_results: list[dict] = []

    async def drain() -> None:
        async for batch in run.events():
            for event in batch.events():
                if event.kind not in ("effect_completed", "effect_failed"):
                    continue
                body = json.loads(event.to_json())["body"][event.kind]
                if body.get("output_contract", {}).get("kind") != "tool_result":
                    continue
                if event.kind == "effect_failed":
                    # A tool that fails leaves no result to read, and the
                    # model usually just apologises. Print the reason so a
                    # rejected model name or argument is visible here.
                    print(f"  tool call failed: {body['error']}")
                    continue
                for item in body["output"]["content"]:
                    if item.get("kind") == "json":
                        tool_results.append(item["value"])

    drained = asyncio.create_task(drain())
    started = time.monotonic()
    deadline = started + timeout_seconds
    next_beat = started + 15.0
    approvals = 0
    result = None
    while time.monotonic() < deadline:
        pending = await run.list_interactions()
        if pending:
            approvals += 1
            # Every approval is a separately billed call. Approve up to the
            # budget and deny the rest, so a model that retries instead of
            # reading its own tool result cannot run up a bill.
            approved = approvals <= budget
            verdict = "approving" if approved else "denying repeat"
            print(f"  {verdict} paid call #{approvals}")
            await run.resolve_interaction(approval_resolution(pending[0], approved))
            continue
        try:
            result = await asyncio.wait_for(run.result(), timeout=1.0)
            break
        except asyncio.TimeoutError:
            if heartbeat and time.monotonic() >= next_beat:
                print(f"  [{time.monotonic() - started:5.0f}s] {heartbeat}")
                next_beat = time.monotonic() + 15.0
            await asyncio.sleep(0.1)
    await run.close_events()
    await drained
    if result is None:
        raise TimeoutError("run did not reach a terminal state in time")
    return result, tool_results


SUFFIXES = {"image/jpeg": ".jpg", "image/png": ".png", "audio/mpeg": ".mp3"}


def save_media(
    handle: "finstack_ai.Agent", tool_results: list[dict], stem: str
) -> pathlib.Path:
    """Write the first staged artifact in `tool_results` to `MEDIA_DIR`.

    The extension follows the artifact's own media type rather than the
    caller's guess, because the tool decides what it produced: the same
    speech tool returns `audio/mpeg` or raw `audio/pcm` depending on the
    format its caller asked for.
    """

    for value in tool_results:
        artifact = value.get("artifact")
        if artifact is None:
            continue
        media_type = value["media_type"]
        suffix = (
            SUFFIXES.get(media_type) or "." + media_type.split(";")[0].split("/")[-1]
        )
        MEDIA_DIR.mkdir(exist_ok=True)
        path = (MEDIA_DIR / (stem + suffix)).resolve()
        path.write_bytes(handle.read_artifact(artifact))
        return path
    raise LookupError("no staged artifact in this run's tool results")

### 1. Text

The plain `agent` from the construction cell, no tools and no media. A
rejected or revoked key fails the run here, before any model output
begins.

In [ ]:
if api_key:
    text_result = await agent.run("What is 20 plus 22? Reply with only the number.")
    print("text:", text_result.text)
else:
    print("skipped: OPENROUTER_API_KEY unset")

### 2. Text plus tools

A Python `@finstack_ai.tool` function reaches the model as an OpenRouter
tool. Local Python tools carry no approval requirement, so this runs
through `Agent.run` in one shot; only the paid OpenRouter media tools
park on an approval interaction.

In [ ]:
from pydantic import BaseModel


class Answer(BaseModel):
    answer: int


@finstack_ai.tool
def add(left: int, right: int) -> Answer:
    """Add two integers."""
    return Answer(answer=left + right)


tools = finstack_ai.pydantic_toolset(
    add,
    component="notebook.toolset.openrouter",
    name="math",
)

if api_key:
    tool_agent = await finstack_ai.Agent.openrouter(
        OPENROUTER_MODEL,
        "Answer with the tool result only.",
        api_key=api_key,
        reasoning_effort=OPENROUTER_REASONING_EFFORT or None,
        reasoning_summary=OPENROUTER_REASONING_SUMMARY or None,
        toolsets=[tools],
    )
    tool_result = await tool_agent.run("Add 20 and 22")
    print("tool:", tool_result.text)
else:
    print("skipped: OPENROUTER_API_KEY unset")

### 3. Images

`openrouter_generate_image` renders one image and stages it. Name an
image model explicitly: OpenRouter's image catalogue
(`GET /api/v1/images/models`) is separate from its chat catalogue, and
`model` is a required tool argument with no default. Content filters
differ sharply between image models, so a prompt one model refuses
another renders without complaint.

The bytes come back as an artifact reference rather than in
`result.text`, so the image URL below is a local `file://` URL for the
file `save_media` just wrote, not a remote one.

In [ ]:
if api_key:
    image_result, image_tools = await run_with_approvals(
        media_agent,
        "Generate one image of a serene mountain landscape at sunset with "
        "dramatic clouds, using the bytedance-seed/seedream-5-0-pro model. "
        "Call openrouter_generate_image exactly once, then confirm it is done.",
    )
    print("image:", image_result.text)
    image_path = save_media(media_agent, image_tools, "openrouter-image")
    print("image url:", image_path.as_uri())
else:
    print("skipped: OPENROUTER_API_KEY unset")

### 4. Video

`openrouter_generate_video` submits a job and returns a job id
immediately; `openrouter_get_video` polls that id and takes a
`wait_seconds` argument (0-300) that keeps polling inside OpenRouter
until the job finishes or the budget is spent. Both tools authenticate
independently and bill to the media toolset's own key, whichever
provider serves the chat model.

Video is the one modality with a real remote URL: OpenRouter hosts the
finished clip and returns it under `urls` in the poll result, so this
cell prints that URL instead of writing a local file.

Rendering takes 30 seconds to several minutes, so the prompt asks for a
single `openrouter_get_video` call with `wait_seconds=300` rather than
spending a model turn per poll, and this cell prints a heartbeat while
it waits. Keep the prompt to attributes the tool schema accepts: asking
for a detail it cannot express invites a resubmit to satisfy the
request, and every submit is a separately billed job. The approval
budget of 1 makes that bounded rather than merely unlikely.

In [ ]:
if api_key:
    video_result, video_tools = await run_with_approvals(
        media_agent,
        "Generate a four-second video of a red circle on a white background "
        "using the bytedance/seedance-2.0-mini model. Call "
        "openrouter_generate_video exactly once, then call openrouter_get_video "
        "once with that job id and wait_seconds=300, which waits for the job "
        "inside that single call. Only if it still returns pending or "
        "in_progress, call openrouter_get_video again with the same id. Never "
        "submit a second video job. Report the final status and the download URL.",
        timeout_seconds=600.0,
        heartbeat="still rendering (measured ~100-150s)",
    )
    print("video:", video_result.text)
    for value in video_tools:
        for url in value.get("urls") or []:
            print("video url:", url)
else:
    print("skipped: OPENROUTER_API_KEY unset")

### 5. Speech to text

`openrouter_transcribe_audio` takes an `audio_url`, downloads it inside
the tool, and posts the bytes to OpenRouter's
`/api/v1/audio/transcriptions` endpoint. The transcript is text, so it
arrives in the tool result and in `result.text` directly — this is the
one media tool with no artifact to read back.

The download sends a `User-Agent` header, which public hosts such as
Wikimedia require; without one they answer `403` and the tool reports a
download failure rather than a transcription failure.

In [ ]:
CLIP_URL = "https://upload.wikimedia.org/wikipedia/commons/1/1f/George_W_Bush_Columbia_FINAL.ogg"

if api_key:
    stt_result, _ = await run_with_approvals(
        media_agent,
        "Transcribe the audio at "
        f"{CLIP_URL} using the openai/whisper-1 model. Call "
        "openrouter_transcribe_audio exactly once with that url, then report "
        "the first sentence of the transcript.",
    )
    print("speech to text:", stt_result.text)
    print("source url:", CLIP_URL)
else:
    print("skipped: OPENROUTER_API_KEY unset")

### 6. Text to speech

`openrouter_generate_speech` posts to `/api/v1/audio/speech` and returns
audio bytes, which are staged exactly like the generated image. Name a
speech model and a voice it accepts: OpenRouter's speech models are a
third catalogue, listed by neither `GET /api/v1/models` nor the chat
models whose `output_modalities` include `audio`, and `voice` names do
not carry across models — `alloy` on `x-ai/grok-voice-tts-1.0` is a
provider `404`, while `eve` renders.

As with the image, there is no remote URL to report — the printed URL
points at the local MP3 that `save_media` wrote, which most notebook
front ends will open in a player.

In [ ]:
if api_key:
    tts_result, tts_tools = await run_with_approvals(
        media_agent,
        "Say 'Hello from the OpenRouter media toolset' as speech using the "
        "x-ai/grok-voice-tts-1.0 model with the eve voice. Call "
        "openrouter_generate_speech exactly once, then confirm it is done.",
    )
    print("text to speech:", tts_result.text)
    speech_path = save_media(media_agent, tts_tools, "openrouter-speech")
    print("speech url:", speech_path.as_uri())
else:
    print("skipped: OPENROUTER_API_KEY unset")